# Two-Voice Podcast Generator

Clone your voice (and a second host's) and turn scripts into finished narration audio.

**Before you start:** set the runtime to a GPU — *Runtime > Change runtime type > T4 GPU*. A free T4 is enough for the 1.5B model.

Run the cells top to bottom. The whole setup takes about 5 minutes the first time.

In [ ]:
#@title 1. Check you actually have a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
print('\nIf that errored: Runtime > Change runtime type > T4 GPU, then rerun.')

In [ ]:
#@title 2. Install VibeVoice (~4 min)
import os

# MIT-licensed community preservation fork of Microsoft's withdrawn repo.
if not os.path.isdir('/content/VibeVoice'):
    !git clone --depth 1 https://github.com/vibevoice-community/VibeVoice.git /content/VibeVoice
!pip install -q -e /content/VibeVoice

# Our wrapper scripts. Point this at your own repo to pull the latest version.
REPO = 'https://github.com/joe8080/Joemoyo-.git'  #@param {type:"string"}
BRANCH = 'claude/custom-voice-generator-7zhdny'    #@param {type:"string"}
if not os.path.isdir('/content/joemoyo'):
    !git clone --depth 1 -b $BRANCH $REPO /content/joemoyo

os.makedirs('/content/joemoyo/voice/voices', exist_ok=True)
os.makedirs('/content/joemoyo/voice/scripts', exist_ok=True)
print('\nInstalled. Restart the runtime only if pip asks you to.')

## 3. Upload your voice samples

You need one clip per speaker: **15-25 seconds of clean, continuous speech**, no music, no second voice, no room echo. An MP4 straight off your phone is fine — the next cell strips the audio and converts it.

Read the sample the way you actually narrate. The clip sets the register for the entire episode.

In [ ]:
#@title Upload and convert a voice sample
#@markdown Run this once per speaker, changing the name and gender each time.
NAME = 'Joe'      #@param {type:"string"}
GENDER = 'man'    #@param ["man", "woman"]
START_SECONDS = 0    #@param {type:"number"}
DURATION_SECONDS = 20 #@param {type:"number"}

from google.colab import files
up = files.upload()
src = '/content/' + list(up.keys())[0]

!cd /content/joemoyo && python voice/prepare_voice.py \
  --input "$src" --name $NAME --gender $GENDER \
  --start $START_SECONDS --duration $DURATION_SECONDS --force

In [ ]:
#@title Listen back before you commit to it
import glob
from IPython.display import Audio, display
for wav in sorted(glob.glob('/content/joemoyo/voice/voices/*.wav')):
    print(wav.split('/')[-1])
    display(Audio(wav))

## 4. Write the script

Use `Speaker 1:` and `Speaker 2:` labels — Speaker 1 gets the first voice you list, Speaker 2 the second.

You can also paste raw output from ScriptWriterAgent with `JOE:` / `MAYA:` labels and markdown in it; the generator strips stage directions in brackets and normalises the labels automatically.

In [ ]:
#@title Paste your script here
script = """
Speaker 1: In 1961, a plane went down over what was then Northern Rhodesia. On board was Dag Hammarskjold, the Secretary-General of the United Nations.
Speaker 2: And the official verdict at the time was pilot error.
Speaker 1: Pilot error. That was the finding. But three separate inquiries have reopened the case since.
"""

with open('/content/joemoyo/voice/scripts/episode.txt', 'w') as f:
    f.write(script.strip())

# Preview the normalised version without loading the model.
!cd /content/joemoyo && python voice/generate.py \
  --script voice/scripts/episode.txt --voices Joe Maya --dry-run

In [ ]:
#@title 5. Generate
#@markdown First run downloads ~3 GB of model weights. Later runs skip that.
VOICES = 'Joe Maya'  #@param {type:"string"}
MODEL = '1.5b'       #@param ["1.5b", "7b"]
CFG_SCALE = 1.3      #@param {type:"number"}

!cd /content/joemoyo && python voice/generate.py \
  --script voice/scripts/episode.txt \
  --voices $VOICES --model $MODEL --cfg-scale $CFG_SCALE

In [ ]:
#@title 6. Listen and download
from IPython.display import Audio, display
from google.colab import files
out = '/content/joemoyo/voice/outputs/episode.wav'
display(Audio(out))
files.download(out)

## Tuning

| Problem | Fix |
|---|---|
| Doesn't sound enough like you | Better sample — quieter room, 20s, your real narration energy |
| Flat or monotone | Raise `CFG_SCALE` to 1.5-2.0 |
| Warbly, drifting, artefacts | Lower `CFG_SCALE` to 1.1-1.3 |
| Two speakers sound alike | Make the samples more distinct; check you passed two different names |
| Out of memory | Stay on `1.5b`; the `7b` model needs more VRAM than a free T4 has |
| Rushed pacing | Punctuation drives rhythm — use full stops and paragraph breaks, not commas |

Colab disconnects on idle. For a feature-length episode, run it in chunks (the
generator already splits long scripts) or move to a rented GPU with the
Dockerfile in `voice/`.